# 🚀 Unsloth Fine-Tuning Pipeline: Đà Nẵng & Quảng Nam Wedding Tráp Advisor
### Mô-đun Google Colab (QLoRA) Fine-Tune Qwen2.5-7B-Instruct & Export GGUF sang Hugging Face Hub

Hệ thống tự động:
1. Pull dataset mới nhất (`dataset_train.jsonl`, `dataset_val.jsonl`) từ GitHub Repository.
2. Fine-tune mô hình `unsloth/Qwen2.5-7B-Instruct-bnb-4bit` tối ưu trên GPU T4 / A100 bằng Unsloth (Nhanh gấp 2-5x, giảm 80% VRAM).
3. Export mô hình sang định dạng **GGUF** (`q4_k_m` & `f16`).
4. Đẩy trực tiếp mô hình GGUF lên **Hugging Face Hub**.

In [ ]:
# 1. KIỂM TRA GPU VÀ CÀI ĐẶT THƯ VIỆN UNSLOTH + DEPENDENCIES
!nvidia-smi

import torch
gpu_stats = torch.cuda.get_device_properties(0)
print(f"GPU: {gpu_stats.name} | Total VRAM: {gpu_stats.total_memory / 1024**3:.2f} GB")

# Install Unsloth and essential training dependencies
!pip install --no-deps "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps xformers "trl<0.9.0" peft accelerate bitsandbytes
!pip install datasets huggingface_hub gitpython pydantic

In [ ]:
# 2. CLONE HOẶC PULL DATASET MỚI NHẤT TỪ GITHUB REPOSITORY
import os

GITHUB_REPO_URL = "https://github.com/nvt-10-10/train-chatbot.git"
DATA_DIR = "./data"

if os.path.exists("./dataset_repo"):
    print("Updating repository...")
    !cd ./dataset_repo && git pull
else:
    print("Cloning dataset repository...")
    !git clone {GITHUB_REPO_URL} ./dataset_repo || true

# Kiểm tra file dataset_train.jsonl và dataset_val.jsonl
TRAIN_FILE = "./dataset_repo/data/dataset_train.jsonl" if os.path.exists("./dataset_repo/data/dataset_train.jsonl") else "./data/dataset_train.jsonl"
VAL_FILE = "./dataset_repo/data/dataset_val.jsonl" if os.path.exists("./dataset_repo/data/dataset_val.jsonl") else "./data/dataset_val.jsonl"

print(f"Train path: {TRAIN_FILE} (Exists: {os.path.exists(TRAIN_FILE)})")
print(f"Val path: {VAL_FILE} (Exists: {os.path.exists(VAL_FILE)})")

In [ ]:
# 3. LOAD MÔ HÌNH QWEN2.5-7B VÀ THIẾT LẬP QLORA VỚI UNSLOTH
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048  # Độ dài chuỗi tối đa
dtype = None           # Auto detect (Float16 cho T4, Bfloat16 cho Ampere/A100)
load_in_4bit = True    # 4-bit quantization giảm VRAM

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-7B-Instruct-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

# Thiết lập LoRA Target Modules
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,               # LoRA Rank
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,     # Tối ưu 0 dropout cho Unsloth fast patching
    bias = "none",
    use_gradient_checkpointing = "unsloth", # Giảm bớt 30% VRAM
    random_state = 3407,
    use_rssl = False,
    loftq_config = None,
)
print("✅ Mô hình Qwen2.5 QLoRA đã sẵn sàng!")

In [ ]:
# 4. CHUẨN HÓA DỮ LIỆU SANG QWEN2.5 CHAT TEMPLATE
from datasets import load_dataset
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(
    tokenizer,
    chat_template = "qwen-2.5",
)

def formatting_prompts_func(examples):
    convs = examples["messages"]
    texts = [tokenizer.apply_chat_template(convo, tokenize = False, add_generation_prompt = False) for convo in convs]
    return { "text" : texts, }

dataset_train = load_dataset("json", data_files=TRAIN_FILE, split="train")
dataset_val = load_dataset("json", data_files=VAL_FILE, split="train") if os.path.exists(VAL_FILE) else None

dataset_train = dataset_train.map(formatting_prompts_func, batched = True)
if dataset_val:
    dataset_val = dataset_val.map(formatting_prompts_func, batched = True)

print("Sample formatted text:")
print(dataset_train[0]["text"][:400])

In [ ]:
# 5. TIẾN HÀNH FINE-TUNE VỚI SFTTRAINER
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset_train,
    eval_dataset = dataset_val,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False, # Phù hợp cho hội thoại chat
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 60, # Tăng số bước nếu có nhiều sample hơn (vd: 120-300)
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

trainer_stats = trainer.train()
print(f"Training finished in {trainer_stats.metrics['train_runtime']:.2f} seconds!")

In [ ]:
# 6. THỬ NGHIỆM INFERENCE MÔ HÌNH VỪA TRAIN
FastLanguageModel.for_inference(model)

messages = [
    {"role": "system", "content": "Bạn là Chuyên viên Tư vấn Tráp Cưới cao cấp Đà Nẵng & Quảng Nam."},
    {"role": "user", "content": "Dạ shop ơi, em ở bên Hội An muốn hỏi đặt Bộ 5 Tráp Rồng Phượng giao về nhà thì ship thế nào ạ? Với em muốn đem bánh su xê riêng tới shop đơm giúp được không?"},
]
inputs = tokenizer.apply_chat_template(
    messages,
    tokenize = True,
    add_generation_prompt = True,
    return_tensors = "pt"
).to("cuda")

outputs = model.generate(input_ids = inputs, max_new_tokens = 512, use_cache = True)
response = tokenizer.batch_decode(outputs)
print("--- BÀI THỬ NGHIỆM TƯ VẤN ---")
print(response[0].split("<|im_start|>assistant")[1].replace("<|im_end|>", "").strip())

In [ ]:
# 7. EXPORT MÔ HÌNH SANG ĐỊNH DẠNG GGUF (Q4_K_M & F16)
# Unsloth tự động convert GGUF chỉ với 1 dòng lệnh!

# Export Quantized Q4_K_M GGUF (chạy nhẹ trên Ollama local)
model.save_pretrained_gguf("model_gguf_q4", tokenizer, quantization_method = "q4_k_m")

# Hoặc export Float16 GGUF
# model.save_pretrained_gguf("model_gguf_f16", tokenizer, quantization_method = "f16")

In [ ]:
# 8. UPLOAD MÔ HÌNH GGUF TRỰC TIẾP LÊN HUGGING FACE HUB
from huggingface_hub import HfApi, login

HF_TOKEN = "hf_XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX" # Điền Hugging Face Token (Write Permission)
HF_REPO_ID = "YOUR_HF_USERNAME/qwen2.5-7b-trap-danang-quangnam-gguf" # Tên repo HF của bạn

if HF_TOKEN.startswith("hf_"):
    login(token=HF_TOKEN)
    api = HfApi()
    api.create_repo(repo_id=HF_REPO_ID, repo_type="model", exist_ok=True)
    
    # Push folder GGUF lên Hugging Face Hub
    model.push_to_hub_gguf(
        repo_id = HF_REPO_ID,
        tokenizer = tokenizer,
        quantization_method = "q4_k_m",
        token = HF_TOKEN,
    )
    print(f"🎉 Đã upload thành công mô hình GGUF lên Hugging Face Hub: https://huggingface.co/{HF_REPO_ID}")
else:
    print("⚠️ Chưa nhập HF_TOKEN. Vui lòng nhập token có quyền Write để đẩy mô hình lên Hugging Face Hub.")